# S13 – HuggingFace: инференс готовой BERT-модели для классификации текста

В этом ноутбуке переходим от **подготовки входов** к следующему естественному шагу:  
используем **уже обученную модель классификации** и смотрим, как она работает на реальных текстах.

Главная цель ноутбука – показать два уровня работы с HuggingFace:

- **быстрый high-level путь** через `pipeline`;
- **явный low-level путь** через `AutoTokenizer` и `AutoModelForSequenceClassification`.

После этого должно быть понятно:

- как загрузить готовую модель классификации текста;
- что такое `logits`, вероятности, `label` и `score`;
- чем удобен `pipeline` и что скрывается у него внутри;
- как выполнить инференс для одного текста и для батча;
- почему готовую модель нельзя бездумно применять к любой задаче и любому домену.

В качестве демо-модели используем **`nlptown/bert-base-multilingual-uncased-sentiment`**.  
Это мультиязычная BERT-модель для анализа тональности, которая возвращает оценки от `1 star` до `5 stars`.

Такой выбор удобен по двум причинам:

1. модель BERT-подобная и хорошо подходит для знакомства с inference-пайплайном;
2. пространство меток здесь интуитивно понятно: от явно негативной к явно позитивной оценке.

Важно: это **готовая модель общего назначения**, а не модель, обученная специально под наши учебные тексты.  
Поэтому часть примеров будет предсказываться неидеально – и это полезная часть семинара, а не ошибка постановки.


## 0. План

К концу ноутбука надо уметь:

1. Загружать готовую BERT-модель для sequence classification.
2. Запускать быстрый инференс через `pipeline`.
3. Получать предсказания вручную через `AutoTokenizer` и `AutoModelForSequenceClassification`.
4. Разбирать `logits`, вероятности и итоговый класс.
5. Выполнять батчевый инференс для списка текстов.
6. Сравнивать уверенные и пограничные предсказания.
7. Понимать ограничения готовой модели и видеть, когда нужен fine-tuning под свою задачу.


## 1. Импорты и общие настройки

Ниже подключаем библиотеки, фиксируем `seed`, выбираем устройство и загружаем готовую модель классификации.

> Предполагается, что в окружении уже доступны `torch`, `transformers`, `pandas` и `numpy`.


In [ ]:
# Базовые библиотеки для воспроизводимости, работы с массивами и удобного вывода результатов.
import random
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    pipeline,
)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)
pd.set_option("display.precision", 4)

In [ ]:
# Фиксируем seed и определяем устройство.
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pipeline_device = 0 if torch.cuda.is_available() else -1
print(f"Device: {device}")

# Готовая мультиязычная BERT-модель для sentiment classification.
MODEL_NAME = "nlptown/bert-base-multilingual-uncased-sentiment"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

print("Model loaded:", MODEL_NAME)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Model class:", model.__class__.__name__)
print("Number of labels:", model.config.num_labels)
print("id2label:", model.config.id2label)

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded: nlptown/bert-base-multilingual-uncased-sentiment
Tokenizer class: BertTokenizer
Model class: BertForSequenceClassification
Number of labels: 5
id2label: {0: '1 star', 1: '2 stars', 2: '3 stars', 3: '4 stars', 4: '5 stars'}


## 2. Модель и учебный набор текстов

Для демонстрации нам нужен небольшой набор русскоязычных примеров:

- несколько **явно позитивных**;
- несколько **явно негативных**;
- несколько **пограничных или смешанных**;
- несколько **не совсем подходящих под домен отзывов**, чтобы увидеть ограничения модели.

Это важно: модель обучалась не на наших учебных формулировках, поэтому технические, официальные или неоднозначные тексты могут обрабатываться хуже, чем типичные отзывы.


In [ ]:
# Набор учебных текстов: часть очевидные, часть пограничные, часть вне типичного домена отзывов.
demo_texts = [
    "Этот курс оказался очень полезным, объяснения понятные и примеры удачные.",
    "Полный провал: приложение зависает, данные теряются, пользоваться невозможно.",
    "В целом нормально, но документация местами запутанная и примеров маловато.",
    "Интерфейс аккуратный, всё работает быстро, результатом доволен.",
    "Ожидал большего: идея хорошая, но реализация сырая и нестабильная.",
    "Модель успешно обработала запрос и вернула ответ без ошибок.",
    "Сервис формально выполнил задачу, однако процесс занял слишком много времени.",
    "Не могу сказать, что продукт плохой, но и особого восторга он не вызвал.",
]

pd.DataFrame({"text": demo_texts})

,text
0,"Этот курс оказался очень полезным, объяснения понятные и примеры удачные."
1,"Полный провал: приложение зависает, данные теряются, пользоваться невозможно."
2,"В целом нормально, но документация местами запутанная и примеров маловато."
3,"Интерфейс аккуратный, всё работает быстро, результатом доволен."
4,"Ожидал большего: идея хорошая, но реализация сырая и нестабильная."
5,Модель успешно обработала запрос и вернула ответ без ошибок.
6,"Сервис формально выполнил задачу, однако процесс занял слишком много времени."
7,"Не могу сказать, что продукт плохой, но и особого восторга он не вызвал."


## 3. Быстрый старт через `pipeline`

`pipeline` – самый удобный способ быстро проверить готовую модель на нескольких примерах.

Преимущества:

- минимум кода;
- не нужно руками собирать входы;
- сразу получаем человекочитаемое предсказание.

Но у этого удобства есть цена: часть шагов скрыта внутри.  
Именно поэтому позже мы повторим тот же инференс вручную.


In [ ]:
# Создаём high-level pipeline для классификации текста.
text_clf = pipeline(
    task="text-classification",
    model=model,
    tokenizer=tokenizer,
    device=pipeline_device,
)

pipeline_outputs = text_clf(demo_texts)

pipeline_df = pd.DataFrame(
    {
        "text": demo_texts,
        "predicted_label": [item["label"] for item in pipeline_outputs],
        "score": [item["score"] for item in pipeline_outputs],
    }
)

pipeline_df

,text,predicted_label,score
0,"Этот курс оказался очень полезным, объяснения понятные и примеры удачные.",4 stars,0.5202
1,"Полный провал: приложение зависает, данные теряются, пользоваться невозможно.",1 star,0.8299
2,"В целом нормально, но документация местами запутанная и примеров маловато.",3 stars,0.4645
3,"Интерфейс аккуратный, всё работает быстро, результатом доволен.",4 stars,0.4757
4,"Ожидал большего: идея хорошая, но реализация сырая и нестабильная.",2 stars,0.5569
5,Модель успешно обработала запрос и вернула ответ без ошибок.,5 stars,0.5819
6,"Сервис формально выполнил задачу, однако процесс занял слишком много времени.",2 stars,0.3856
7,"Не могу сказать, что продукт плохой, но и особого восторга он не вызвал.",2 stars,0.4168


## 4. Что означают `label` и `score`

У выбранной модели пространство меток соответствует рейтингу от `1 star` до `5 stars`.

Грубая интерпретация такая:

- `1 star` – явно негативный текст;
- `2 stars` – скорее негативный;
- `3 stars` – нейтральный или смешанный;
- `4 stars` – скорее позитивный;
- `5 stars` – явно позитивный.

Поле `score` в выводе `pipeline` – это не «истинная вероятность корректности», а уверенность модели в выбранной метке.  
Эта величина полезна, но интерпретировать её надо осторожно.


In [ ]:
# Добавим более удобную интерпретацию предсказаний.
def stars_to_sentiment(label: str) -> str:
    if label.startswith("1") or label.startswith("2"):
        return "negative"
    if label.startswith("3"):
        return "mixed / neutral"
    if label.startswith("4") or label.startswith("5"):
        return "positive"
    return "unknown"

pipeline_df["sentiment_group"] = pipeline_df["predicted_label"].apply(stars_to_sentiment)
pipeline_df = pipeline_df[["text", "predicted_label", "sentiment_group", "score"]]
pipeline_df

,text,predicted_label,sentiment_group,score
0,"Этот курс оказался очень полезным, объяснения понятные и примеры удачные.",4 stars,positive,0.5202
1,"Полный провал: приложение зависает, данные теряются, пользоваться невозможно.",1 star,negative,0.8299
2,"В целом нормально, но документация местами запутанная и примеров маловато.",3 stars,mixed / neutral,0.4645
3,"Интерфейс аккуратный, всё работает быстро, результатом доволен.",4 stars,positive,0.4757
4,"Ожидал большего: идея хорошая, но реализация сырая и нестабильная.",2 stars,negative,0.5569
5,Модель успешно обработала запрос и вернула ответ без ошибок.,5 stars,positive,0.5819
6,"Сервис формально выполнил задачу, однако процесс занял слишком много времени.",2 stars,negative,0.3856
7,"Не могу сказать, что продукт плохой, но и особого восторга он не вызвал.",2 stars,negative,0.4168


## 5. Тот же инференс вручную: `AutoTokenizer` + `AutoModelForSequenceClassification`

Теперь повторим тот же процесс более явно.

Это важный шаг, потому что именно здесь видно, из каких стадий состоит inference:

1. текст превращается в тензоры токенизатором;
2. модель возвращает `logits`;
3. к `logits` применяется `softmax`;
4. по вероятностям выбирается наиболее вероятный класс.

Такой путь более многословен, но он снимает эффект «магии» у `pipeline`.


In [ ]:
# Вспомогательная функция для ручного инференса одного текста.
def predict_one_text(text: str) -> Dict[str, Any]:
    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
    )
    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)

    logits = outputs.logits
    probs = F.softmax(logits, dim=-1)

    pred_id = int(torch.argmax(probs, dim=-1).item())
    pred_label = model.config.id2label[pred_id]
    pred_score = float(probs[0, pred_id].item())

    result = {
        "text": text,
        "pred_id": pred_id,
        "pred_label": pred_label,
        "pred_score": pred_score,
        "logits": logits.cpu().numpy().round(4).tolist()[0],
        "probabilities": probs.cpu().numpy().round(4).tolist()[0],
    }
    return result

single_example = demo_texts[0]
manual_result = predict_one_text(single_example)

pd.DataFrame(
    {
        "field": list(manual_result.keys()),
        "value": [manual_result[key] for key in manual_result.keys()],
    }
)

,field,value
0,text,"Этот курс оказался очень полезным, объяснения понятные и примеры удачные."
1,pred_id,3
2,pred_label,4 stars
3,pred_score,0.5202
4,logits,"[-3.1310999393463135, -2.4684998989105225, 0.3944999873638153, 2.396699905395508, 2.1424999237060547]"
5,probabilities,"[0.002099999925121665, 0.004000000189989805, 0.07029999792575836, 0.5202000141143799, 0.4034999907016754]"


## 6. Смотрим полное распределение вероятностей

`pipeline` по умолчанию обычно показывает только лучший класс.  
Ручной вызов удобен тем, что позволяет увидеть **все вероятности по всем меткам**.

Это особенно полезно для пограничных текстов:  
модель может выбрать один класс, но при этом иметь довольно близкие вероятности для соседних меток.


In [ ]:
# Удобная функция для преобразования вероятностей в таблицу.
def probability_table(text: str) -> pd.DataFrame:
    result = predict_one_text(text)
    labels = [model.config.id2label[i] for i in range(model.config.num_labels)]
    df = pd.DataFrame(
        {
            "label": labels,
            "probability": result["probabilities"],
        }
    ).sort_values("probability", ascending=False, ignore_index=True)
    df.insert(0, "text", text)
    return df

probability_table(demo_texts[2])

,text,label,probability
0,"В целом нормально, но документация местами запутанная и примеров маловато.",3 stars,0.4645
1,"В целом нормально, но документация местами запутанная и примеров маловато.",2 stars,0.4039
2,"В целом нормально, но документация местами запутанная и примеров маловато.",1 star,0.0863
3,"В целом нормально, но документация местами запутанная и примеров маловато.",4 stars,0.0425
4,"В целом нормально, но документация местами запутанная и примеров маловато.",5 stars,0.0027


## 7. Сравнение `pipeline` и ручного инференса

Ниже прогоним весь учебный набор двумя способами и убедимся, что итоговые предсказания совпадают.

Это важная проверка:  
`pipeline` – не другая модель, а удобная обёртка над тем же самым inference-процессом.


In [ ]:
# Батчевый ручной инференс для списка текстов.
def predict_many_texts(texts: List[str]) -> pd.DataFrame:
    encoded = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
    )
    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)

    probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()
    pred_ids = probs.argmax(axis=1)

    rows = []
    for text, pred_id, prob_row in zip(texts, pred_ids, probs):
        pred_label = model.config.id2label[int(pred_id)]
        pred_score = float(prob_row[int(pred_id)])
        row = {
            "text": text,
            "manual_label": pred_label,
            "manual_score": pred_score,
        }
        for i in range(model.config.num_labels):
            row[model.config.id2label[i]] = float(prob_row[i])
        rows.append(row)

    return pd.DataFrame(rows)

manual_df = predict_many_texts(demo_texts)
comparison_df = pipeline_df.merge(manual_df[["text", "manual_label", "manual_score"]], on="text")
comparison_df["same_label"] = comparison_df["predicted_label"] == comparison_df["manual_label"]
comparison_df

,text,predicted_label,sentiment_group,score,manual_label,manual_score,same_label
0,"Этот курс оказался очень полезным, объяснения понятные и примеры удачные.",4 stars,positive,0.5202,4 stars,0.5202,True
1,"Полный провал: приложение зависает, данные теряются, пользоваться невозможно.",1 star,negative,0.8299,1 star,0.8299,True
2,"В целом нормально, но документация местами запутанная и примеров маловато.",3 stars,mixed / neutral,0.4645,3 stars,0.4645,True
3,"Интерфейс аккуратный, всё работает быстро, результатом доволен.",4 stars,positive,0.4757,4 stars,0.4757,True
4,"Ожидал большего: идея хорошая, но реализация сырая и нестабильная.",2 stars,negative,0.5569,2 stars,0.5569,True
5,Модель успешно обработала запрос и вернула ответ без ошибок.,5 stars,positive,0.5819,5 stars,0.5819,True
6,"Сервис формально выполнил задачу, однако процесс занял слишком много времени.",2 stars,negative,0.3856,2 stars,0.3856,True
7,"Не могу сказать, что продукт плохой, но и особого восторга он не вызвал.",2 stars,negative,0.4168,2 stars,0.4168,True


## 8. Батчевый инференс и удобная сводная таблица

При реальной эксплуатации мы почти всегда обрабатываем не один текст, а список текстов.  
Поэтому полезно иметь табличное представление, где видно:

- сам текст;
- выбранную метку;
- уверенность модели;
- распределение по всем классам.

Именно такой формат потом легко адаптируется под CSV, API или потоковую обработку.


In [ ]:
# Упорядочим столбцы так, чтобы было удобно читать распределение по всем звёздам.
ordered_columns = [
    "text",
    "manual_label",
    "manual_score",
    "1 star",
    "2 stars",
    "3 stars",
    "4 stars",
    "5 stars",
]

manual_df[ordered_columns]

,text,manual_label,manual_score,1 star,2 stars,3 stars,4 stars,5 stars
0,"Этот курс оказался очень полезным, объяснения понятные и примеры удачные.",4 stars,0.5202,0.0021,0.0040,0.0703,0.5202,0.4035
1,"Полный провал: приложение зависает, данные теряются, пользоваться невозможно.",1 star,0.8299,0.8299,0.1567,0.0119,0.0009,0.0005
2,"В целом нормально, но документация местами запутанная и примеров маловато.",3 stars,0.4645,0.0863,0.4039,0.4645,0.0425,0.0027
3,"Интерфейс аккуратный, всё работает быстро, результатом доволен.",4 stars,0.4757,0.0021,0.0052,0.0861,0.4757,0.4309
4,"Ожидал большего: идея хорошая, но реализация сырая и нестабильная.",2 stars,0.5569,0.0927,0.5569,0.3361,0.0130,0.0013
5,Модель успешно обработала запрос и вернула ответ без ошибок.,5 stars,0.5819,0.0191,0.0161,0.0484,0.3345,0.5819
6,"Сервис формально выполнил задачу, однако процесс занял слишком много времени.",2 stars,0.3856,0.1602,0.3856,0.3732,0.0721,0.0090
7,"Не могу сказать, что продукт плохой, но и особого восторга он не вызвал.",2 stars,0.4168,0.3926,0.4168,0.1553,0.0273,0.0080


## 9. Что такое `logits` и почему нужен `softmax`

Модель sequence classification возвращает **сырые оценки классов**, которые называются `logits`.

Сами по себе `logits` неудобны для интерпретации:

- они могут быть отрицательными;
- они не обязаны суммироваться к 1;
- их нельзя напрямую читать как вероятности.

Поэтому после модели обычно применяют `softmax`, чтобы получить нормированное распределение вероятностей.

**Формула softmax** для класса $i$ при наличии $K$ классов:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

где $z_i$ – это `logit` класса $i$.

Главные свойства после softmax:
- все значения строго больше 0;
- сумма по всем классам равна 1;
- наибольший `logit` даёт наибольшую вероятность, порядок классов сохраняется.

> В ячейке ниже это хорошо видно: `logits` могут быть отрицательными, а вероятности после softmax – уже нет.


In [ ]:
# Посмотрим на logits и probabilities для одного примера более явно.
example_text = "Интерфейс приятный, но система периодически зависает и это раздражает."
encoded = tokenizer(example_text, return_tensors="pt", truncation=True, padding=True)
encoded = {k: v.to(device) for k, v in encoded.items()}

with torch.no_grad():
    outputs = model(**encoded)

logits = outputs.logits.cpu().numpy()[0]
probabilities = F.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
labels = [model.config.id2label[i] for i in range(model.config.num_labels)]

pd.DataFrame(
    {
        "label": labels,
        "logit": np.round(logits, 4),
        "probability": np.round(probabilities, 4),
    }
).sort_values("probability", ascending=False, ignore_index=True)

,label,logit,probability
0,3 stars,1.8315,0.5812
1,2 stars,0.7524,0.1976
2,4 stars,0.6619,0.1805
3,1 star,-1.2958,0.0255
4,5 stars,-1.8081,0.0153


## 10. Стресс-тест на сложных и пограничных примерах

Теперь специально подадим модели примеры, на которых ей потенциально трудно:

- отрицание;
- смешанная оценка;
- сухой технический текст;
- формально корректный, но эмоционально неочевидный текст;
- саркастические или почти саркастические формулировки.

Это нужно не для «поймать модель на ошибке», а чтобы увидеть реальные пределы готового решения.


In [ ]:
# Пограничные и неудобные примеры.
stress_texts = [
    "Неплохо, хотя ожидания были заметно выше.",
    "Не могу сказать, что я доволен результатом.",
    "Функция заявлена, но фактически в рабочем сценарии не помогает.",
    "Ну да, конечно, великолепный сервис: снова всё упало в пятницу вечером.",
    "Система завершила обработку запроса за 0.23 секунды.",
    "Выглядит красиво, но пользы на практике почти нет.",
    "Это не лучший продукт в мире, но работает стабильно и без сюрпризов.",
    "Совсем не плохо.",
]

stress_df = predict_many_texts(stress_texts)
stress_df["sentiment_group"] = stress_df["manual_label"].apply(stars_to_sentiment)
stress_df[["text", "manual_label", "sentiment_group", "manual_score", "1 star", "2 stars", "3 stars", "4 stars", "5 stars"]]

,text,manual_label,sentiment_group,manual_score,1 star,2 stars,3 stars,4 stars,5 stars
0,"Неплохо, хотя ожидания были заметно выше.",3 stars,mixed / neutral,0.4580,0.0844,0.3588,0.4580,0.0889,0.0099
1,"Не могу сказать, что я доволен результатом.",4 stars,positive,0.4436,0.0118,0.0229,0.1670,0.4436,0.3547
2,"Функция заявлена, но фактически в рабочем сценарии не помогает.",1 star,negative,0.4126,0.4126,0.4082,0.1596,0.0169,0.0027
3,"Ну да, конечно, великолепный сервис: снова всё упало в пятницу вечером.",3 stars,mixed / neutral,0.3249,0.1268,0.1955,0.3249,0.2268,0.1261
4,Система завершила обработку запроса за 0.23 секунды.,1 star,negative,0.5773,0.5773,0.2417,0.1122,0.0449,0.0239
5,"Выглядит красиво, но пользы на практике почти нет.",3 stars,mixed / neutral,0.4614,0.0825,0.3590,0.4614,0.0892,0.0079
6,"Это не лучший продукт в мире, но работает стабильно и без сюрпризов.",4 stars,positive,0.5807,0.0031,0.0172,0.3051,0.5807,0.0938
7,Совсем не плохо.,3 stars,mixed / neutral,0.2659,0.1294,0.1903,0.2659,0.2142,0.2002


## 11. Несколько наблюдений по стресс-тесту

Обычно на таких примерах быстро проявляются типичные особенности готовых моделей:

1. **Отрицание и двойное отрицание** могут интерпретироваться не так, как ожидает человек.
2. **Сарказм и ирония** остаются трудной областью для большинства моделей.
3. **Технические нейтральные тексты** часто насильно притягиваются к ближайшей эмоциональной метке.
4. **Смешанные отзывы** могут давать распределение между `3 stars` и соседними классами.

То есть уже здесь видно: готовая модель полезна для быстрого старта, но не гарантирует хорошее качество вне своего домена.


## 12. Ограничения готовой модели

Важно зафиксировать, чего **не стоит ожидать** от предобученной модели общего назначения.

### 12.1. Ограничение по домену
Модель обучалась на определённом типе текстов.  
Если подать юридические, медицинские, академические, технические или очень формальные тексты, поведение может стать менее надёжным.

### 12.2. Ограничение по пространству меток
Эта модель умеет предсказывать только шкалу `1 star ... 5 stars`.  
Если нам нужны классы вида `spam / not spam`, `bug / feature request`, `urgent / non-urgent`, понадобится другая модель или fine-tuning.

### 12.3. Ограничение по языку и стилю
Хотя модель мультиязычная, качество на конкретном языке и конкретном стиле текста может быть неоднородным.

### 12.4. Ограничение по интерпретации score
Высокий `score` – это не гарантия истинности, а лишь сильное предпочтение модели в рамках её собственного пространства классов.

Именно поэтому следующим естественным шагом становится **тонкая настройка BERT под свою прикладную задачу**.


## 13. Итоги

1. **Готовую BERT-модель можно быстро применить через `pipeline`.**
2. **Тот же инференс можно выполнить вручную через `AutoTokenizer` и `AutoModelForSequenceClassification`.**
3. **Модель возвращает `logits`, а после `softmax` мы получаем вероятности по классам.**
4. **Батчевый инференс – это стандартный и удобный режим работы.**
5. **Пограничные и вне-доменные тексты сразу показывают пределы готового решения.**
6. **Если метки и данные у нас свои, дальше нужен fine-tuning, а не слепое использование общей модели.**


## Задания для самостоятельной работы

1. Возьмите 10 собственных русскоязычных текстов и проверьте, какие из них модель оценивает уверенно, а какие – нет.
2. Найдите 5 примеров с отрицанием (`неплохо`, `не могу сказать, что доволен`, `совсем не плохо`) и сравните распределения вероятностей.
3. Подайте в модель тексты из другого домена: технические логи, формальные письма, фрагменты документации. Посмотрите, как меняются предсказания.
4. Реализуйте собственную функцию, которая по вероятностям сводит `1-2 stars` в `negative`, `3 stars` в `neutral`, `4-5 stars` в `positive`.
5. Сравните вывод `pipeline` и ручного инференса ещё на одном наборе примеров и убедитесь, что итоговые метки совпадают.
